# Mathematical Basics - Part 2

This notebook covers the material in the last three chapters of the CWJSN Mathematical Basics section. Since much of this material is also covered in the Math_Foundations repo, I will only add interesting new stuff I learn along the way.

In [1]:
# Set up Revise.jl for automatic code reloading
# This only needs to be run once at the beginning of your notebook session
using Revise

# Load the package 
using Calculus
using Latexify

## Inequalities and Logical Expressions Chapter

### Geometric Mean expressed in Julia

It is not clear why Verzani chose in CWJSN to use the expression `(a * b)^(1/2)` in the example about the geometric mean, rather than `sqrt(a * b)` as the code for the expression $g = \sqrt{ab}$. A related question is whether `sqrt(a * b) == (a * b)^(1/2)` is always `true`. I asked Claude to provide some insight on this.

The geometric mean of $n$ numbers is

$$
G = \left(x_1 x_2 \cdots x_n\right)^{1/n}, \qquad n = 2:\ G = (ab)^{1/2} = \sqrt{ab},
$$

so the power form shows the definition and generalizes by changing the exponent. In the same CWJSN code block, the next line uses `sqrt` for the quadratic mean, so the choice may be stylistic.

| Measured (Julia 1.13.2, $10^7$ values of `rand()`) | Result |
| --- | --- |
| `x^(1/2) != sqrt(x)` | about 1.05% of values |
| largest difference | 1 ulp (unit in the last place) |
| example $x$ | `0.7092888978478461` |
| `x^(1/2)` | `0.8421929101149249` |
| `sqrt(x)` (correctly rounded) | `0.842192910114925` |

As for whether the two different expressions are `==`, running a test returns `false` for some inputs. With `a = 0.7092888978478461` and `b = 1.0` (chosen so the product will be equal to `x` in the table above):

```julia
sqrt(a * b) == (a * b)^(1/2)   # false
sqrt(a * b)                    # 0.842192910114925
(a * b)^(1/2)                  # 0.8421929101149249
```

Over a million pairs from `rand(2)`, 9,650 gave `false`, about 1 in 100.

**Why the two results differ:**

- **The exponent isn't the cause.** `1/2` is exactly `0.5` in binary, so `(a*b)^(1/2)` really does ask for the power 0.5.
- **`sqrt` always returns the closest floating-point number to the true root.** The IEEE floating-point standard requires that of square root, and the processor has an instruction for it.
- **`^` with a `Float64` exponent uses a general power routine.** It is built to be accurate to within about 1 unit in the last place for any exponent, but it doesn't promise the closest number. About 1% of the time it lands one step away from what `sqrt` gives.

Both results are within 1 unit in the last place of the true root, so `sqrt(a * b) ≈ (a * b)^(1/2)` is always `true`. It's the same `==` versus `≈` lesson as in the Part 1 section "Overflow, rounding and `≈`": two different routes to the same real number can land on adjacent floats.

### The bitwise operators

Verzani mentions bitwise operators but he doesn't explain them at all. He just says a few words about their precedence. So I asked Claude to write something up about bitwise operators. An extensively edited (by me) version follows.

#### What "bitwise" means

An integer is stored as a row of binary bits. For the purpose of bitwise operation, each bit is assigned a logical value. The bit `1` is equivalent to `true` and the bit `0` to `false`. Bitwise operators apply a logical function to the bits. Binary bitwise operators combine two rows of bits, operating on the two matched bits, position by position.

The bitwise operators `&` and `|` apply the truth value function of `and` and `or` respectively. Hence `&` puts a 1 only where both bits are 1, and `|` puts a 1 where either bit is 1.

Using the function `bitstring` that shows the bits in the rows, we get the following:

```julia
bitstring(Int8(12))        # "00001100"
bitstring(Int8(10))        # "00001010"
12 & 10                    # 8   = 00001000
12 | 10                    # 14  = 00001110
```
On truth values, `&` and `|` apply the same truth value function as `&&` and `||`, but they differ from them in four ways discussed below.


#### 1. Both sides are always evaluated

`&&` and `||` stop evaluating expressions as soon as the answer is known, which is why they are called shortcut (short-circuit) operators. `&` and `|` always evaluate both sides first.

A *guard* is a condition placed first so that the expression after it runs only when that is safe. In `x >= 0 && sqrt(x) > 1`, the test `x >= 0` guards `sqrt(x)`: when `x` is negative, `&&` already knows the answer is `false` and never calls `sqrt`. So a guard only works with `&&` (and `||`), not with `&` (or `|`):

```julia
x = -4
x >= 0 && sqrt(x) > 1        # false: sqrt is never called
(x >= 0) & (sqrt(x) > 1)     # DomainError with -4.0: sqrt runs anyway
```

#### 2. Precedence: above the comparisons, not below

`Base.operator_precedence` gives the levels (higher binds tighter):

```julia
Base.operator_precedence.([:&, :*, :|, :+, :<, :&&, :||])   # [12, 12, 11, 11, 7, 6, 5]
```

So `&` sits with `*` and `|` with `+`, both above the comparisons (`<` and the rest, 7), while `&&` and `||` sit below them.

`&&` and `||` bind more loosely than the comparisons, so `x > 0 && x < 10` needs no parentheses. `&` and `|` bind more tightly than the comparisons, so Julia joins `0 & x` before it compares anything:

```julia
x = 20
x > 0 & x < 10          # true  -- parsed as x > (0 & x) < 10, a chained comparison
(x > 0) & (x < 10)      # false -- the intended "0 < x < 10"
```

This gives no error, just a wrong answer. The same thing happens with CWJSN's $\lvert x-5\rvert > 7$:

```julia
x = 18
x - 5 < -7 | x - 5 > 7        # false -- parsed as x - 5 < ((-7 | x) - 5) > 7
(x - 5 < -7) | (x - 5 > 7)    # true
```

**Rule:** with `&` and `|`, put every comparison in parentheses.


#### 3. What the operands may be

`Bool` is Julia's type for truth values. It has exactly two values, `true` and `false`, and a comparison such as `3 > 2` returns one (`typeof(3 > 2)` is `Bool`). `Bool` is a subtype of `Integer`, with `true == 1` and `false == 0`, so arithmetic works on it (`true + true` is `2`). It is stored in a whole byte, not a single bit: `bitstring(true)` is `"00000001"`.

`&&` and `||` require a `Bool` on the left (`1 && true` is a TypeError: non-boolean (Int64) used in boolean context). The right side is returned as it is, so `true && 5` returns `5`. `&` and `|` are integer operations, so `true & 5` returns `1` (an `Int`), and `5 & 3.0` is a MethodError: there is no bitwise "and" for floats.

#### 4. Where `&` and `|` are the right tool: vectors

`&&` asks for one `true` or `false`, so it refuses a vector of them:

```julia
v = [-3, 2, 5, 12]
(v .> 0) && (v .< 10)       # TypeError: non-boolean (BitVector) used in boolean context
(v .> 0) .& (v .< 10)       # Bool[0, 1, 1, 0]
v[(v .> 0) .& (v .< 10)]    # [2, 5] -- keep the entries between 0 and 10
(v .< 0) .| (v .> 10)       # Bool[1, 0, 0, 1]
```
**What `v[(v .> 0) .& (v .< 10)]` does**

The expression listed in the code above needs some explanation. First, it is read from the inside out, with `v = [-3, 2, 5, 12]`. So step by step we see:

```julia
v .> 0                     # Bool[0, 1, 1, 1]  is each entry > 0?
v .< 10                    # Bool[1, 1, 1, 0]  is each entry < 10?
(v .> 0) .& (v .< 10)      # Bool[0, 1, 1, 0]  "and", applied entry by entry
v[Bool[0, 1, 1, 0]]        # [2, 5]            keep the entries marked true
```

1. **The dot broadcasts.** `v .> 0` compares each entry of `v` with 0 and gives one truth value per entry. The result is a `BitVector`, a vector of truth values packed one per bit, which prints as `Bool[…]`.
2. **`.&` applies "and" to each pair.** Position by position it combines the two vectors: true only where the entry is both > 0 and < 10.
3. **Indexing with a vector of truth values selects.** When the index is a vector of `true`/`false` the same length as `v`, `v[mask]` returns the entries where the mask is `true`, in order. This is called *logical indexing*. The mask must have the same length as `v`; `v[[true, false]]` is a BoundsError.

It is the same as finding the positions first and then indexing with them: `findall(mask)` gives `[2, 3]`, and `v[[2, 3]]` gives `[2, 5]`.

Two other ways to write the same selection:

```julia
v[0 .< v .< 10]                  # [2, 5]  the chained comparison, broadcast
filter(x -> 0 < x < 10, v)       # [2, 5]  keep the x for which the test is true
```
**Is it nonetheless possible to use `&&` on vectors? Only the dotted version: `.&&`**

Since Julia 1.7 there are also `.&&` and `.||`. They have the low precedence of `&&`, so `v .> 0 .&& v .< 10` is correct without parentheses (`Bool[0, 1, 1, 0]`).

Plain `&&` still refuses a vector. By contrast, the dotted `.&&` applies `&&` to each matched pair of entries. This is indicated by the Julia 1.7 release notes which state: "The short-circuiting operators `&&` and `||` can now be dotted to participate in broadcast fusion as `.&&` and `.||`." We can see this in action:

```julia
v = [-3, 2, 5, 12]
(v .> 0) && (v .< 10)      # TypeError: non-boolean (BitVector) used in boolean context
v .> 0 .&& v .< 10         # Bool[0, 1, 1, 0]
```

The short-circuit survives, entry by entry, because the whole dotted expression is fused into one loop:

```julia
x = [-4, 4]
x .>= 0 .&& sqrt.(x) .> 1          # Bool[0, 1] -- sqrt(-4) is never called
(x .>= 0) .& (sqrt.(x) .> 1)       # DomainError -- sqrt.(x) runs on every entry first
```

The Julia manual ("Noteworthy differences from other languages", MATLAB section) recommends `.&&` and `.||` for logic over arrays, e.g. `A .== 1 .|| A .== 2`.


#### `missing`

Both kinds of logical operators use the same three-valued logic, and the result is `missing` only when the unknown value could change the answer. `&` and `|` accept `missing` on either side, but `&&` and `||` accept it only on the right:

```julia
true  & missing      # missing -- depends on the unknown
false & missing      # false   -- false whatever the unknown is
true  | missing      # true
false | missing      # missing
missing && true      # TypeError: non-boolean (Missing) used in boolean context
true && missing      # missing -- the right side is returned as it is
```


#### The other bitwise operators

The manual ("Mathematical Operations", "Bitwise Operators") lists nine operators for integers: `~`, `&`, `|`, `⊻`, `⊼`, `⊽`, `>>>`, `>>`, `<<`. Using the same two rows as before (as `Int8`, so that `bitstring` shows 8 bits):

```julia
a, b = Int8(12), Int8(10)   # 00001100, 00001010
~a           # -13  = 11110011   not: flip every bit
a ⊻ b        #   6  = 00000110   xor: 1 where the bits differ
a ⊼ b        #  -9  = 11110111   nand: not (a & b)
a ⊽ b        # -15  = 11110001   nor:  not (a | b)
```

`⊻` is typed `\xor` Tab, `⊼` is `\nand` Tab and `⊽` is `\nor` Tab. Each one also has a name you can call: `xor(a, b)`, `nand(a, b)`, `nor(a, b)`.

Flipping every bit gives a negative number because of how integers store their sign (two's complement): `~x == -x - 1` for every integer, so `~5` is `-6`.

#### On truth values

```julia
~true            # false -- the same as !true
~5               # -6
!5               # MethodError: ! is only for Bool
true ⊻ true      # false -- "exactly one of them is true"
true ⊻ false     # true  (on truth values, ⊻ applies the same truth-value function as !=)
true ⊼ true      # false
false ⊽ false    # true
true ⊻ missing   # missing
```

So `!` is the logical "not" and only takes a `Bool`, while `~` is the bitwise "not" and takes any integer. Like `&` and `|`, all of these evaluate both sides and broadcast with a dot: `[true, false] .⊻ [true, true]` gives `Bool[0, 1]`, and `.~[true, false]` gives `Bool[0, 1]`.

#### Shifts

A shift slides the whole row of bits left or right. Bits that fall off the end are lost.

```julia
1 << 3              # 8   -- shifting left by n multiplies by 2^n
5 >> 1              # 2   -- shifting right by n divides by 2^n, rounding down
-16 >> 2            # -4  -- >> copies the sign bit into the new places (arithmetic shift)
Int8(-16) >>> 2     # 60  -- >>> fills them with 0s (logical shift): 11110000 → 00111100
Int8(64) << 1       # -128 -- the 1 slides into the sign bit: overflow wraps, as with *
1 << 64             # 0   -- every bit has fallen off an Int64
```


#### Precedence of all of them

```julia
Base.operator_precedence.([:^, :<<, :>>, :>>>, :*, :&, :⊼, :+, :|, :⊻, :⊽, :<])
# [15, 14, 14, 14, 12, 12, 12, 11, 11, 11, 11, 7]
```

So the shifts bind more tightly than `*` but less tightly than `^`, the "and" family (`&`, `⊼`) sits with `*`, and the "or" family (`|`, `⊻`, `⊽`) sits with `+`. In C the shifts bind more loosely than `+`, so code copied from C can change meaning: in Julia, `1 << 2 + 1` is `(1 << 2) + 1 = 5`, while in C it is `1 << 3 = 8`. All of these sit above the comparisons. Hence the rule from the previous section applies to every one of them: **_put each comparison in parentheses_**.